## The performance of a lock complex
In this notebook, we simulate a lock object on a network. Randomly generated, evenly sized vessels sampled from an exponential distribution have to pass this lock. We add a complex lock object to the graph. Vessels are levelled in the same lock operation if they can fit inside the lock, and register with the lock operator before the start of the lock operation. Based on this behaviour, we will quantify the performance of the lock in terms of efficiency and capacity.

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import transform

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim

# import of basic core mixins and utils for creating objects, inspecting the output and plotting
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core.utils import (
    create_object, 
    generate_vessels_from_distribution, 
    inspect_object, 
    validate_generation_inputs, 
    list_distributions, 
    create_distribution,
)
from opentnsim.core.visualizations import generate_vessel_gantt_chart

# import of graph
from opentnsim.graph import Graph

# import of simulation
from opentnsim.model import Simulation

# import of modules important for locking
from opentnsim.lock import IsLockChamber, IsLockWaitingArea, IsLockComplex, LockComplexTraversable
from opentnsim.lock.calculations import estimate_lock_capacity, calculate_lock_occupancy
from opentnsim.lock.logutils import calculate_cycle_information, get_vessel_delays

# package(s) needed for inspecting the output
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tqdm

# package(s) needed for running OpenTNSim simulations in parallel
import os
from dask.distributed import Client, LocalCluster
from dask.distributed import as_completed

import logging
# Suppress Dask Distributed INFO messages
logging.getLogger("distributed").setLevel(logging.ERROR)
logging.getLogger("dask").setLevel(logging.ERROR)


print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 2.3.2a2.dev3+g514a752b4.d20260603


#### 0. Create environment

In [2]:
def create_environment(simulation_start = pd.Timestamp('2026-01-01 00:00:00')):
    simulation = Simulation(simulation_start = simulation_start)
    return simulation

#### 1. Create graph
We create a directional graph with a layout similar to that of notebooks 0201, 0202, and 0203. However, we add two additional edges prior to nodes 0 and 1 to show that a lock can also be implemented in a larger graph. These edges are 20km long.

In [3]:
def create_graph_and_add_to_simulation(
    simulation,
    distance_lock_edge = 10000,
):
    graph = Graph()
    graph.add_node(name = '0', geometry = Point(-distance_lock_edge/2,0), epsg = 4087)
    graph.add_node(name = '1', geometry = Point(distance_lock_edge/2,0), epsg = 4087)
    graph.add_edge(('0','1'))
    simulation.add_graph(graph)
    return simulation

#### 1+ Adding infrastructure
We construct the same lock complex as before.

In [4]:
def create_lock_complex_and_add_to_simulation(
    simulation,
    lock_length = 400,
    lock_width = 50,
    lock_depth = 10,
    loop_distance = 500,
):
    env = simulation.env
    lock_chamber = IsLockChamber(
        env=env,
        lock_depth = lock_depth,
        name='Lock',
        gate_open = '0',
        edge = ('0','1'),
        geometry_m = Polygon([
            Point(-lock_length/2, -lock_width/2),
            Point(-lock_length/2, lock_width/2),
            Point(lock_length/2, lock_width/2),
            Point(lock_length/2, -lock_width/2)
        ]),
    )

    edge_length = env.graph.edges[('0','1')]['length_m']
    distance_from_edge_start = edge_length/2-lock_length/2-loop_distance
    
    waiting_area_A = IsLockWaitingArea(
        env=env,
        name = 'Waiting area A',
        edge = ('0','1'),
        orientation = 0,
        distance_from_edge_start = distance_from_edge_start,
    )

    waiting_area_B = IsLockWaitingArea(
        env=env,
        name = 'Waiting area B',
        edge = ('1','0'),
        orientation = 1,
        distance_from_edge_start = distance_from_edge_start,
    )

    lock_complex = IsLockComplex(
        lock_chambers = [lock_chamber],
        waiting_areas = [waiting_area_A, waiting_area_B],
        registration_nodes = ['0','1'],
        env=env,
        name = 'Lock complex',
    )

    return simulation, lock_chamber

#### 2. Create agents
We create the same vessel agent object.

In [5]:
seed = 12345678

In [6]:
Vessel = create_object(
    "Vessel", 
    (
        LockComplexTraversable,     # allows to interact with a lock
    ), 
)

In [28]:
fleet_composition = pd.DataFrame([
    {"vessel_class": "BII-4", "fleet_probability": 0.1387},
    {"vessel_class": "BII-6l", "fleet_probability": 0.0408},
    {"vessel_class": "BII-6b", "fleet_probability": 0.0163},
    {"vessel_class": "C3b", "fleet_probability": 0.0049},
    {"vessel_class": "C3l", "fleet_probability": 0.0734},
    {"vessel_class": "C4", "fleet_probability": 0.2039},
    {"vessel_class": "M5", "fleet_probability": 0.0082},
    {"vessel_class": "M6", "fleet_probability": 0.0326},
    {"vessel_class": "M7", "fleet_probability": 0.0082},
    {"vessel_class": "M8", "fleet_probability": 0.1387},
    {"vessel_class": "M9", "fleet_probability": 0.0979},
    {"vessel_class": "M10", "fleet_probability": 0.0326},
    {"vessel_class": "M11", "fleet_probability": 0.1305},
    {"vessel_class": "M12", "fleet_probability": 0.0734},
])

fleet_composition["fleet_probability"] /= (
    fleet_composition["fleet_probability"].sum()
)

In [29]:
def calculate_draught(
    loading_factor,
    T_loaded,
    T_unloaded,
):
    return (
        T_unloaded
        + loading_factor * (T_loaded - T_unloaded)
    )

In [30]:
fleet_data = pd.DataFrame([
    # Motorized vessels
    {
        "vessel_class": "M5",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 40, **{'mean': 85, 'std': 2.90}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 40, **{'mean': 8.2, 'std': 0.12}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 40, **{'mean': 2.7, 'std': 0.17}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1040, **{'mean': 1.5, 'std': 0.17}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'Cargo',
        "name": "M5",
    },
    {
        "vessel_class": "M6",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 50, **{'mean': 85, 'std': 6.09}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 50, **{'mean': 9.5, 'std': 0.31}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 50, **{'mean': 2.9, 'std': 0.27}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1050, **{'mean': 1.0, 'std': 0.27}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'Cargo',
        "name": "M6",
    },
    {
        "vessel_class": "M7",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 60, **{'mean': 105, 'std': 6.20}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 60, **{'mean': 9.5, 'std': 0.20}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 60, **{'mean': 3.0, 'std': 0.26}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1060, **{'mean': 1.0, 'std': 0.26}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'Cargo',
        "name": "M7",
    },
    {
        "vessel_class": "M8",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 70, **{'mean': 110, 'std': 11.64}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 70, **{'mean': 11.4, 'std': 0.69}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 70, **{'mean': 3.5, 'std': 0.47}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1070, **{'mean': 1.8, 'std': 0.47}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'Cargo',
        "name": "M8",
    },
    {
        "vessel_class": "M9",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 80, **{'mean': 135, 'std': 15.46}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 80, **{'mean': 11.4, 'std': 0.71}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 80, **{'mean': 3.5, 'std': 0.85}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1080, **{'mean': 1.8, 'std': 0.85}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'Cargo',
        "name": "M9",
    },
    {
        "vessel_class": "M10",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 90, **{'mean': 110, 'std': 12.60}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 90, **{'mean': 13.6, 'std': 0.85}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 90, **{'mean': 4.0, 'std': 1.0}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1090, **{'mean': 2.0, 'std': 1.0}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'Cargo',
        "name": "M10",
    },
    {
        "vessel_class": "M11",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 100, **{'mean': 185, 'std': 21.18}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 100, **{'mean': 14.2, 'std': 0.88}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 100, **{'mean': 4.0, 'std': 1.0}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1100, **{'mean': 2.0, 'std': 1.0}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'Cargo',
        "name": "M11",
    },
    {
        "vessel_class": "M12",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 110, **{'mean': 135, 'std': 15.46}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 110, **{'mean': 17.0, 'std': 1.06}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 110, **{'mean': 4.0, 'std': 1.0}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1110, **{'mean': 2.0, 'std': 1.0}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'Cargo',
        "name": "M12",
    },

    # Push-tow units
    {
        "vessel_class": "BII-4",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 210, **{'mean': 193, 'std': 1.88}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 210, **{'mean': 22.8, 'std': 0.51}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 210, **{'mean': 4.0, 'std': 0.37}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1210, **{'mean': 2.0, 'std': 0.37}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'PushBarge',
        "name": "BII-4",
    },
    {
        "vessel_class": "BII-6l",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 220, **{'mean': 193, 'std': 1.88}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 220, **{'mean': 22.8, 'std': 0.51}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 220, **{'mean': 4.0, 'std': 0.37}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1220, **{'mean': 2.0, 'std': 0.37}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'PushBarge',
        "name": "BII-6l",
    },
    {
        "vessel_class": "BII-6b",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 220, **{'mean': 195, 'std': 1.88}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 220, **{'mean': 34.2, 'std': 0.62}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 220, **{'mean': 4.0, 'std': 0.37}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1220, **{'mean': 2.0, 'std': 0.37}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'PushBarge',
        "name": "BII-6b",
    },

    # Coupled units
    {
        "vessel_class": "C3l",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 280, **{'mean': 180, 'std': 9}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 280, **{'mean': 11.4, 'std': 0.34}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 280, **{'mean': 3.5, 'std': 0.47}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1280, **{'mean': 1.8, 'std': 0.47}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'CoupledVessel',
        "name": "C3l",
    },
    {
        "vessel_class": "C3b",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 290, **{'mean': 105, 'std': 8}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 290, **{'mean': 22.8, 'std': 1.08}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 290, **{'mean': 3.5, 'std': 0.47}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1290, **{'mean': 1.8, 'std': 0.47}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'CoupledVessel',
        "name": "C3b",
    },
    {
        "vessel_class": "C4",
        "vessel_object": Vessel,
        "L": create_distribution('normal', seed=int(str(seed)[0]) + 300, **{'mean': 185, 'std': 8}),
        "B": create_distribution('normal', seed=int(str(seed)[1]) + 300, **{'mean': 22.8, 'std': 0.97}),
        "T_loaded": create_distribution('normal', seed=int(str(seed)[2]) + 300, **{'mean': 3.5, 'std': 0.47}),
        "T_unloaded": create_distribution('normal', seed=int(str(seed)[2]) + 1300, **{'mean': 1.8, 'std': 0.47}),
        "T":calculate_draught,
        "v": create_distribution('constant', **{'value': 3}),
        "type": 'CoupledVessel',
        "name": "C4",
    },
])

In [31]:
correlation_matrices = {

    # Motorized cargo vessels
    "M5": pd.DataFrame(
        [
            [1.0, 0.70, 0.35, 0.50],
            [0.70, 1.0, 0.35, 0.55],
            [0.35, 0.35, 1.0, 0.75],
            [0.50, 0.55, 0.75, 1.0],
        ],
        index=["L", "B", "T_unloaded", "T_loaded"],
        columns=["L", "B", "T_unloaded", "T_loaded"],
    ),

    "M6": pd.DataFrame(
        [
            [1.0, 0.70, 0.35, 0.50],
            [0.70, 1.0, 0.35, 0.55],
            [0.35, 0.35, 1.0, 0.75],
            [0.50, 0.55, 0.75, 1.0],
        ],
        index=["L", "B", "T_unloaded", "T_loaded"],
        columns=["L", "B", "T_unloaded", "T_loaded"],
    ),

    "M7": pd.DataFrame(
        [
            [1.0, 0.70, 0.35, 0.50],
            [0.70, 1.0, 0.35, 0.55],
            [0.35, 0.35, 1.0, 0.75],
            [0.50, 0.55, 0.75, 1.0],
        ],
        index=["L", "B", "T_unloaded", "T_loaded"],
        columns=["L", "B", "T_unloaded", "T_loaded"],
    ),

    "M8": pd.DataFrame(
        [
            [1.0, 0.70, 0.35, 0.50],
            [0.70, 1.0, 0.35, 0.55],
            [0.35, 0.35, 1.0, 0.75],
            [0.50, 0.55, 0.75, 1.0],
        ],
        index=["L", "B", "T_unloaded", "T_loaded"],
        columns=["L", "B", "T_unloaded", "T_loaded"],
    ),

    "M9": pd.DataFrame(
        [
            [1.0, 0.70, 0.35, 0.50],
            [0.70, 1.0, 0.35, 0.55],
            [0.35, 0.35, 1.0, 0.75],
            [0.50, 0.55, 0.75, 1.0],
        ],
        index=["L", "B", "T_unloaded", "T_loaded"],
        columns=["L", "B", "T_unloaded", "T_loaded"],
    ),

    "M10": pd.DataFrame(
        [
            [1.0, 0.70, 0.35, 0.50],
            [0.70, 1.0, 0.35, 0.55],
            [0.35, 0.35, 1.0, 0.75],
            [0.50, 0.55, 0.75, 1.0],
        ],
        index=["L", "B", "T_unloaded", "T_loaded"],
        columns=["L", "B", "T_unloaded", "T_loaded"],
    ),

    "M11": pd.DataFrame(
        [
            [1.0, 0.70, 0.35, 0.50],
            [0.70, 1.0, 0.35, 0.55],
            [0.35, 0.35, 1.0, 0.75],
            [0.50, 0.55, 0.75, 1.0],
        ],
        index=["L", "B", "T_unloaded", "T_loaded"],
        columns=["L", "B", "T_unloaded", "T_loaded"],
    ),

    "M12": pd.DataFrame(
        [
            [1.0, 0.70, 0.35, 0.50],
            [0.70, 1.0, 0.35, 0.55],
            [0.35, 0.35, 1.0, 0.75],
            [0.50, 0.55, 0.75, 1.0],
        ],
        index=["L", "B", "T_unloaded", "T_loaded"],
        columns=["L", "B", "T_unloaded", "T_loaded"],
    ),

    # Push-tow units
    "BII-4": pd.DataFrame(
        [
            [1.0, 0.65, 0.25, 0.40],
            [0.65, 1.0, 0.30, 0.50],
            [0.25, 0.30, 1.0, 0.70],
            [0.40, 0.50, 0.70, 1.0],
        ],
        index=["L", "B", "T_unloaded", "T_loaded"],
        columns=["L", "B", "T_unloaded", "T_loaded"],
    ),

    "BII-6b": pd.DataFrame(
        [
            [1.0, 0.65, 0.25, 0.40],
            [0.65, 1.0, 0.30, 0.50],
            [0.25, 0.30, 1.0, 0.70],
            [0.40, 0.50, 0.70, 1.0],
        ],
        index=["L", "B", "T_unloaded", "T_loaded"],
        columns=["L", "B", "T_unloaded", "T_loaded"],
    ),

    # Coupled units
    "C3l": pd.DataFrame(
        [
            [1.0, 0.75, 0.30, 0.45],
            [0.75, 1.0, 0.35, 0.60],
            [0.30, 0.35, 1.0, 0.75],
            [0.45, 0.60, 0.75, 1.0],
        ],
        index=["L", "B", "T_unloaded", "T_loaded"],
        columns=["L", "B", "T_unloaded", "T_loaded"],
    ),

    "C3b": pd.DataFrame(
        [
            [1.0, 0.75, 0.30, 0.45],
            [0.75, 1.0, 0.35, 0.60],
            [0.30, 0.35, 1.0, 0.75],
            [0.45, 0.60, 0.75, 1.0],
        ],
        index=["L", "B", "T_unloaded", "T_loaded"],
        columns=["L", "B", "T_unloaded", "T_loaded"],
    ),

    "C4": pd.DataFrame(
        [
            [1.0, 0.75, 0.30, 0.45],
            [0.75, 1.0, 0.35, 0.60],
            [0.30, 0.35, 1.0, 0.75],
            [0.45, 0.60, 0.75, 1.0],
        ],
        index=["L", "B", "T_unloaded", "T_loaded"],
        columns=["L", "B", "T_unloaded", "T_loaded"],
    ),
}

In [32]:
def create_vessels_and_add_to_simulation(
    simulation,
    fleet_composition,
    fleet_data,
    correlation_matrices,
    mean_arrival_rate_up,
    mean_arrival_rate_down,
    number_of_vessels_up,
    number_of_vessels_down,
    cv_up = 0.5,
    cv_down = 0.5,
):
    env = simulation.env

    def mission(env, vessel):
        """
        Method that defines the mission of the vessel.
        
        In this case: 
            keep moving along the path until its end point is reached
        """
        while True:
            yield from vessel.move()
            
            if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
                break
    
    # We use the above info to generate upstream and downstream vessels: the interarrival times are generated according to a gamma distribution.
    upstream_vessels = generate_vessels_from_distribution(
        env=env,
        fleet_data=fleet_data,
        fleet_composition=fleet_composition,
        arrival_distribution=create_distribution('gamma', seed = int(str(seed)[3]), **{'mean': mean_arrival_rate_up, 'cv': cv_up}), #in [min]
        loading_factor = create_distribution(
            'beta_mixture', seed=int(str(seed)[2]) + 1280, **{'mode_1': 0.1, 'mode_2': 0.9, 'std_1': 0.05, 'std_2': 0.05, 'mixing_weight': 0.5}),
        number_of_vessels=number_of_vessels_up,
        origin='0',
        destination='1',
        fleet_seed=int(str(seed)[4]),
        correlation_matrices = correlation_matrices,
    )
    
    downstream_vessels = generate_vessels_from_distribution(
        env=env,
        fleet_data=fleet_data,
        fleet_composition=fleet_composition,
        arrival_distribution=create_distribution('gamma', seed = int(str(seed)[3]) + 10, **{'mean': mean_arrival_rate_down, 'cv': cv_down}), #in [min]
        loading_factor = create_distribution(
            'beta_mixture', seed=int(str(seed)[2]) + 1280, **{'mode_1': 0.1, 'mode_2': 0.9, 'std_1': 0.05, 'std_2': 0.05, 'mixing_weight': 0.5}),
        number_of_vessels=number_of_vessels_down,
        origin='0',
        destination='1',
        fleet_seed=int(str(seed)[4]) + 10,
        correlation_matrices = correlation_matrices,
    )
    
    vessels = upstream_vessels + downstream_vessels
    
    for vessel in vessels:
        simulation.add_vessel(vessel, mission)

    return simulation

#### 3. Run simulation
We run the simulation.

In [33]:
def run_simulation(
    lock_length,
    fleet_composition,
    fleet_data,
    correlation_matrices,
    nr_vessels,
    mean_arrival_rate,
):
    simulation = create_environment()
    simulation = create_graph_and_add_to_simulation(simulation)
    simulation, lock_chamber = create_lock_complex_and_add_to_simulation(simulation, lock_length)
    simulation = create_vessels_and_add_to_simulation(
        simulation, 
        fleet_composition,
        fleet_data,
        correlation_matrices,
        mean_arrival_rate, 
        mean_arrival_rate, 
        nr_vessels, 
        nr_vessels)
    simulation.run()
    
    summary = lock_chamber.get_performance(show=False)

    fig = lock_chamber.plot(
        xlimmin = -6050, 
        xlimmax = 6050,
        ylimmin = pd.Timestamp('2026-01-01 00:00:00'),
        ylimmax = pd.Timestamp('2026-01-01 10:00:00'),
        method='Plotly',
        boundary_nodes = ['0','1']
    )
    
    return summary, fig

In [34]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [35]:
summary, fig = run_simulation(
    400,
    fleet_composition,
    fleet_data,
    correlation_matrices,
    1,
    40,
)

{'L': np.float64(218.5732223428606),
 'B': np.float64(14.55145933947496),
 'T': np.float64(3.6525040758935328),
 'v': np.float64(3.0),
 'type': 'Cargo',
 'name': 'M11 0',
 'arrival_time': Timestamp('2026-01-01 00:25:32.901828106'),
 'env': <opentnsim.model.SimulationEnvironment at 0x186e68fbb30>,
 'origin': '0',
 'destination': '1',
 'route': None}

{'L': np.float64(132.43500791484107),
 'B': np.float64(12.750125353880009),
 'T': np.float64(3.615714182585357),
 'v': np.float64(3.0),
 'type': 'Cargo',
 'name': 'M9 1',
 'arrival_time': Timestamp('2026-01-01 00:51:39.745561176'),
 'env': <opentnsim.model.SimulationEnvironment at 0x186e68fbb30>,
 'origin': '0',
 'destination': '1',
 'route': None}

In [ ]:
cpu_cores = os.cpu_count() or 2
n_workers = max(1, min(cpu_cores - 1, 8))

In [ ]:
cpu_cores = os.cpu_count() or 2
n_workers = max(1, min(cpu_cores - 1, 8))

cluster = LocalCluster(
    n_workers=n_workers,
    threads_per_worker=1,
    processes=True,
    dashboard_address=":8787"
)

client = Client(cluster)
client

In [ ]:
lock_length_min = 200
lock_length_max = 600
dL = 25
lock_lengths = np.arange(lock_length_min, lock_length_max + dL, dL)

futures = {
    client.submit(
        run_simulation, 
            lock_length,
            fleet_composition = fleet_composition,
            fleet_data = fleet_data,
            correlation_matrices = correlation_matrices,
            nr_vessels = 100,
            mean_arrival_rate = 60,
    ): lock_length
    for lock_length in lock_lengths
}

summaries = {}
figures = {}

for future in as_completed(futures):
    length = futures[future]
    results = future.result()
    summaries[length] = results[0]
    figures[length] = results[1]

In [ ]:
client.close()
cluster.close()

#### 4. Inspect output
We skip the logbooks and inspect the Gantt charts and time-distance diagram.

In [ ]:
locking_delays = {}
waiting_to_enter_delays = {}
waiting_for_next_locking_cycle_delays = {}
ic_ratios = {}
for lock_length, summary in summaries.items():
    ic_ratios[lock_length] = summary['Cycle-averaged I/C-ratio']
    locking_delays[lock_length] = ((
        summary['Delay causes']['obstruction (%)'] + summary['Delay causes']['operation of lock (%)']
    )/100)*summary['Average individual vessel delay']
    waiting_to_enter_delays[lock_length] = (summary['Delay causes']['traffic (%)']/100)*summary['Average individual vessel delay']
    waiting_for_next_locking_cycle_delays[lock_length] = (summary['Delay causes']['congestion (%)']/100)*summary['Average individual vessel delay']

In [ ]:
fig = figures[200]
fig

In [ ]:
fig = figures[300]
fig

In [ ]:
fig = figures[600]
fig

In [ ]:
# Extract x values
x = sorted(locking_delays.keys())

# Convert Timedelta to hours
y1 = [locking_delays[k].total_seconds()/3600 for k in x]
y2 = [waiting_to_enter_delays[k].total_seconds()/3600 for k in x]
y3 = [waiting_for_next_locking_cycle_delays[k].total_seconds()/3600 for k in x]

# Create stacked bar plot
fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(x, y1, label='Locking', color='lightgreen', width = 10)
ax.bar(x, y2, bottom=y1, label='Waiting to enter', color='orange', width = 10)
ax.bar(x, y3, bottom=[a+b for a, b in zip(y1, y2)],
       label='Waiting for next locking cycle', color='red', width = 10)
ylims = ax.get_ylim()

for i, ic_ratio in ic_ratios.items():
    ax.text(
        i,
        ylims[-1],  # small offset above bar
        f'{ic_ratio:.2f}',
        ha='center',
        va='bottom',
        fontsize=8
    )

ax.set_xlabel('Lock length (m)')
plt.ylabel('Waiting time (hours)')

plt.xticks(x, rotation=45)
plt.legend()
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from opentnsim.core.utils import inspect_object

In [ ]:
Vessel = create_object(
    "Vessel", 
    (
        LockComplexTraversable,     # allows to interact with a lock
    ), 
)

In [ ]:
inspect_object(Vessel)